# Tiled CUDA Matrix Multiplication – Week 02

**What this notebook does:** We compile and run two CUDA kernels for matrix multiplication (C = A × B):
1. **Naive** – each thread reads directly from global memory
2. **Tiled** – threads cooperate using `__shared__` memory to cache tiles

**Concepts you'll learn:**
- Why shared-memory tiling speeds up matrix multiply
- Comparing naive vs tiled CUDA kernels
- PyTorch baseline (`A @ B`)

**Prerequisite:** Enable GPU in Colab (Runtime → Change runtime type → GPU).

## 1. Setup: Clone Repo & Install nvcc

We need the CUDA compiler (`nvcc`) to build our `.cu` file.

In [ ]:
# Clone the repo and go to project root
# %cd is a Jupyter magic to change directory
!git clone https://github.com/HamidrezaGh/llm-performance.git
%cd llm-performance

In [ ]:
# Install nvcc (CUDA compiler) if not found
!apt-get update -qq
!apt-get install -y nvidia-cuda-toolkit 2>/dev/null || true

# Verify GPU and driver version
!nvidia-smi

## 2. Compile the Matrix Multiply Kernels

`matrix_mul.cu` contains:
- **naiveMatMul** – simple baseline, one output element per thread
- **tiledMatMul** – uses 32×32 shared-memory tiles for better data reuse

Matrix size: 2048×2048. Tile size: 32×32.

In [ ]:
# -O3 = optimize for speed
# -Wno-deprecated-gpu-targets = suppress deprecation warnings on newer GPUs
!nvcc -O3 -o matrix_mul 01-kernels/02-cuda-tiled-matmul/matrix_mul.cu -Wno-deprecated-gpu-targets

## 3. Run the CUDA Program

Times both kernels and prints the speedup (tiled vs naive). Expect ~5–7× speedup from tiling.

In [ ]:
!./matrix_mul

## 4. PyTorch Baseline

Same 2048×2048 matrix multiply using PyTorch's `@` operator (highly optimized cuBLAS under the hood).

In [ ]:
!python 01-kernels/02-cuda-tiled-matmul/benchmark.py